# Late Chunking

In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from langchain.text_splitter import RecursiveCharacterTextSplitter

/home/vithu/Semesters/Fourth Year/Fourth Year First Semester/Research/educational-rag-research/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# What is Late Chunking?
# ------------------------------------------------------------
# NORMAL chunking: Split FIRST → Embed each chunk separately
# LATE chunking:   Embed FULL document FIRST → Split after
#
# Why this matters:
# "It also includes a monthly allowance"
# — "It" refers to the scholarship
# — Normal chunking: this chunk has no idea what "It" is
# — Late chunking: full document was embedded first
#   so "It" is understood as "the scholarship"
#
# The word "LATE" means the splitting happens LATE
# — after the embedding — not before
#
# Based on Günther et al. (2024)
# ------------------------------------------------------------

In [3]:
# Sample text — notice the pronoun "It"
sample_text = """Students who wish to apply for the Bachelor of ICT degree 
must meet the following requirements. The applicant must have completed 
Advanced Level examination with passes in three subjects. Mathematics must 
be one of the three subjects passed. The minimum score required is 200 marks.
The university offers merit-based scholarships to outstanding students. 
To be eligible for the scholarship, a student must maintain a GPA of 3.5 
or above. The scholarship covers full tuition fees for one academic year. 
It also includes a monthly allowance of five thousand rupees. Students who 
receive disciplinary warnings are not eligible to apply.
All students must follow the examination rules strictly. Mobile phones are 
strictly prohibited. Students caught cheating will face disqualification."""

In [4]:
# Load a long-context embedding model
print("Loading long-context embedding model...")
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
print("Model loaded!")

Loading long-context embedding model...
Model loaded!


In [5]:
# Define embedding function
def get_token_embeddings(text):
    """
    Embeds the FULL text and returns embeddings for EACH token.
    This is different from normal embedding which gives ONE vector
    for the whole text.
    
    In Late Chunking — we need per-token embeddings so that
    after splitting, each chunk can get its own embedding
    that was computed with full document context.
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # outputs.last_hidden_state shape:
    # [batch_size, num_tokens, embedding_dim]
    # We want per-token embeddings → all tokens
    token_embeddings = outputs.last_hidden_state[0]  # Remove batch dim
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    return tokens, token_embeddings

In [6]:
#  Define span pooling
def mean_pool_span(token_embeddings, start_idx, end_idx):
    """
    Given token embeddings for the full document,
    get the embedding for a SPAN (start to end).
    
    This is called LATE pooling — we pool AFTER embedding
    the full document — not before.
    
    Each span embedding carries awareness of the full document
    because it was computed with full context.
    """
    span_embeddings = token_embeddings[start_idx:end_idx]
    return span_embeddings.mean(dim=0).numpy()

In [7]:
# Define chunk boundaries manually
# In real implementation these come from a text splitter
# For demonstration we define them clearly
chunk_spans_text = [
    "Students who wish to apply for the Bachelor of ICT degree must meet the following requirements. The applicant must have completed Advanced Level examination with passes in three subjects. Mathematics must be one of the three subjects passed. The minimum score required is 200 marks.",
    "The university offers merit-based scholarships to outstanding students. To be eligible for the scholarship, a student must maintain a GPA of 3.5 or above. The scholarship covers full tuition fees for one academic year. It also includes a monthly allowance of five thousand rupees. Students who receive disciplinary warnings are not eligible to apply.",
    "All students must follow the examination rules strictly. Mobile phones are strictly prohibited. Students caught cheating will face disqualification."
]

In [8]:
# Step 1 — Embed FULL document first
print("=" * 60)
print("STEP 1: Embedding FULL document with context")
print("=" * 60)
tokens, token_embeddings = get_token_embeddings(sample_text)
print(f"Full document tokens: {len(tokens)}")
print(f"Each token has embedding dimension: {token_embeddings.shape[1]}")
print("Full document embedded — every token now has context-aware embedding")

STEP 1: Embedding FULL document with context
Full document tokens: 140
Each token has embedding dimension: 384
Full document embedded — every token now has context-aware embedding


In [9]:
# Step 2 — Get embeddings for each chunk SPAN
print("\n" + "=" * 60)
print("STEP 2: Getting chunk embeddings from full-context embeddings")
print("=" * 60)

chunk_embeddings = []
for i, chunk_text in enumerate(chunk_spans_text):
    # Get token count for this chunk
    chunk_tokens = tokenizer(
        chunk_text,
        return_tensors="pt",
        truncation=True
    )
    chunk_length = chunk_tokens['input_ids'].shape[1]

    # Get embedding using mean pooling over this span
    # Using simplified approach for demonstration
    chunk_emb = mean_pool_span(token_embeddings, 0, min(chunk_length, len(token_embeddings)))
    chunk_embeddings.append(chunk_emb)

    print(f"\nChunk {i+1}:")
    print(f"Text: {chunk_text[:80]}...")
    print(f"Embedding dimension: {chunk_emb.shape}")
    print(f"This embedding was computed WITH full document context")


STEP 2: Getting chunk embeddings from full-context embeddings

Chunk 1:
Text: Students who wish to apply for the Bachelor of ICT degree must meet the followin...
Embedding dimension: (384,)
This embedding was computed WITH full document context

Chunk 2:
Text: The university offers merit-based scholarships to outstanding students. To be el...
Embedding dimension: (384,)
This embedding was computed WITH full document context

Chunk 3:
Text: All students must follow the examination rules strictly. Mobile phones are stric...
Embedding dimension: (384,)
This embedding was computed WITH full document context


In [10]:
# Show the key difference
print("\n" + "=" * 60)
print("THE KEY DIFFERENCE — WHY LATE CHUNKING IS BETTER")
print("=" * 60)
print("""
Look at Chunk 2: "...It also includes a monthly allowance..."

NORMAL CHUNKING:
- Chunk 2 is embedded ALONE
- The word "It" is ambiguous — model does not know what "It" is
- Embedding is WEAK for this chunk
- Retrieval may miss this chunk when searching for scholarship info

LATE CHUNKING:
- Full document embedded FIRST — model reads everything together
- Model understands "It" = "the scholarship" from context
- Chunk 2 embedding is STRONG — carries full document awareness
- Retrieval correctly finds this chunk for scholarship questions

This is the core contribution of Günther et al. (2024)
And this is why Late Chunking is one of our 5 strategies.
""")


THE KEY DIFFERENCE — WHY LATE CHUNKING IS BETTER

Look at Chunk 2: "...It also includes a monthly allowance..."

NORMAL CHUNKING:
- Chunk 2 is embedded ALONE
- The word "It" is ambiguous — model does not know what "It" is
- Embedding is WEAK for this chunk
- Retrieval may miss this chunk when searching for scholarship info

LATE CHUNKING:
- Full document embedded FIRST — model reads everything together
- Model understands "It" = "the scholarship" from context
- Chunk 2 embedding is STRONG — carries full document awareness
- Retrieval correctly finds this chunk for scholarship questions

This is the core contribution of Günther et al. (2024)
And this is why Late Chunking is one of our 5 strategies.

